# Model Serving with FastAPI — Solutions

> 📘 **Python Mastery** · Module 18 — MLOps · Lesson 4/6
>
> Complete solutions with explanations for the Model Serving with FastAPI exercises.

## Setup

In [ ]:
# Required imports for all solutions
from fastapi import FastAPI, HTTPException, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, validator
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from typing import List
from datetime import datetime

---
## Part 1: Conceptual Questions

### Solution 1.1: Batch vs Real-Time

**Answer:**

1. **Fraud detection at purchase** → **Real-time API**
   - The decision must happen in milliseconds while the customer waits at checkout
   - Latency budget: sub-second
   - High stakes: approving fraud or declining legitimate purchases directly impacts revenue and customer experience

2. **Monthly churn risk scoring** → **Batch scoring**
   - The campaign is planned in advance; overnight processing is acceptable
   - Latency budget: hours are fine
   - All customers scored once, results written to a table for campaign execution

3. **Ride arrival time estimation** → **Real-time API**
   - Users request rides NOW and expect immediate feedback
   - Latency budget: seconds at most
   - Each request is unique (location, time, traffic) and cannot be pre-computed

4. **Re-scoring historical orders** → **Batch scoring**
   - Retrospective analysis, no user waiting
   - Latency budget: can take hours or even days
   - Process millions of records in parallel against warehouse data

**Key distinction:** Real-time serves individual requests triggered by user actions with tight latency requirements. Batch processes large datasets on a schedule where throughput matters more than per-request latency.

### Solution 1.2: Validation Strategy

**Answer:**

HTTP 422 (Unprocessable Entity) is semantically correct for validation failures:

- **422 means:** "I understood your request syntax, but the content violates the schema/business rules"
- **400 means:** "Your request is malformed — I can't even parse it" (bad JSON, missing headers)
- **500 means:** "I broke — this is MY fault, not yours" (server crash, database down)

**What 422 communicates to clients:**
1. The server is working correctly (not a 500)
2. The request structure was parseable (not a 400)
3. **The client needs to fix their payload** — different field values, not different code
4. The detailed error response tells them exactly what to fix

This distinction helps developers debug: a 422 means "check your data", while a 500 means "check the logs/contact the backend team".

**FastAPI convention:** Pydantic validation failures automatically return 422 with detailed field-level errors, making the API self-documenting.

### Solution 1.3: Model Loading Strategy

**Answer:**

**Approach A is correct.** The model should be loaded at module level (once when the app starts), not inside the endpoint.

**Why Approach A is correct:**
- The model is loaded **once** when the Python module is imported
- Every request reuses the same loaded model in memory
- Fast inference: no I/O overhead per request
- Predictable latency: p99 and p50 latency are similar

**Why Approach B is wrong:**
- Every single request pays the model loading cost
- For a 100MB model, that's 100MB of disk I/O + deserialization per request
- Latency spikes from ~10ms to potentially seconds
- Throughput collapses: you're doing expensive I/O in the request path
- Memory thrashing: repeatedly loading and garbage collecting the same object

**Real-world impact:** A model that can serve 1000 req/sec in Approach A might handle only 1-10 req/sec in Approach B.

**Best practice:** Load all expensive resources (models, feature encoders, database connections) at module level. Endpoints should be thin wrappers that transform input → call model → transform output.

### Solution 1.4: NumPy Type Conversion

**Answer:**

**Why conversion is necessary:**

Python's `json` module (which FastAPI uses) only knows how to serialize built-in types:
- `int`, `float`, `bool`, `str`, `list`, `dict`, `None`

NumPy types like `np.float32`, `np.int64`, `np.bool_` are **different types** that look similar but aren't recognized by the JSON encoder.

**Error without conversion:**
```python
TypeError: Object of type float32 is not JSON serializable
```

**When it happens:**
- `model.predict()` returns `np.ndarray` with dtype like `float32` or `int64`
- `model.predict_proba()` returns `np.ndarray` with `float64`
- Array indexing like `proba[0][1]` gives you `np.float64`, not Python `float`
- Boolean comparisons like `proba >= 0.5` give `np.bool_`

**Solution:**
```python
# Convert scalars
python_float = float(numpy_value)  # np.float32 → float
python_bool = bool(numpy_value)    # np.bool_ → bool
python_int = int(numpy_value)      # np.int64 → int

# Convert arrays
python_list = numpy_array.tolist() # np.ndarray → list
```

**Best practice:** Always convert at the boundary — right before the return statement. Keep NumPy types internally for performance, convert only what crosses the API surface.

---
## Part 2: Coding Exercises

### Solution 2.1: Basic Routes with Path and Query Parameters

In [ ]:
# Solution 2.1
app = FastAPI()


@app.get("/models/{model_id}")
def get_model(model_id: str):
    return {
        "model_id": model_id,
        "status": "active",
        "version": "1.0.0"
    }


@app.get("/models/{model_id}/metrics")
def get_metrics(model_id: str, metric_type: str = "accuracy"):
    return {
        "model_id": model_id,
        "metric_type": metric_type,
        "value": 0.95
    }


# Tests
client = TestClient(app)

# Test 1: Get model info
response = client.get("/models/student-pass-v1")
assert response.status_code == 200
data = response.json()
assert data["model_id"] == "student-pass-v1"
assert data["status"] == "active"
assert data["version"] == "1.0.0"
print("✓ Test 1 passed: GET /models/{model_id}")

# Test 2: Get metrics with default metric_type
response = client.get("/models/student-pass-v1/metrics")
assert response.status_code == 200
data = response.json()
assert data["metric_type"] == "accuracy"  # default
assert "value" in data
print("✓ Test 2 passed: Default metric_type")

# Test 3: Get metrics with custom metric_type
response = client.get("/models/student-pass-v1/metrics?metric_type=f1_score")
assert response.status_code == 200
data = response.json()
assert data["metric_type"] == "f1_score"
print("✓ Test 3 passed: Custom metric_type")

print("\n✅ All tests passed!")

**Explanation:**

- **Path parameters** (`{model_id}`) are declared in the route and function signature
- **Query parameters** (`metric_type`) are function arguments with defaults
- FastAPI automatically parses URLs like `/models/abc/metrics?metric_type=f1_score`
- TestClient follows the same `requests` API: `.get()`, `.post()`, `.json()`, etc.
- No server started, no ports opened — pure in-process testing

### Solution 2.2: Pydantic Request Validation

In [ ]:
# Solution 2.2
class HousingFeatures(BaseModel):
    bedrooms: int = Field(ge=1, le=10)
    bathrooms: float = Field(ge=0.5, le=8.0)
    sqft: float = Field(ge=100, le=10000)
    year_built: int = Field(ge=1800, le=2030)


app = FastAPI()


@app.post("/validate")
def validate_housing(features: HousingFeatures):
    return {"validated": features.model_dump()}


# Tests
client = TestClient(app)

# Test 1: Valid payload
valid_payload = {
    "bedrooms": 3,
    "bathrooms": 2.5,
    "sqft": 1800,
    "year_built": 2010
}
response = client.post("/validate", json=valid_payload)
assert response.status_code == 200
assert response.json()["validated"] == valid_payload
print("✓ Test 1 passed: Valid payload accepted")

# Test 2: Invalid bedrooms (too low)
response = client.post("/validate", json={**valid_payload, "bedrooms": 0})
assert response.status_code == 422
print("✓ Test 2 passed: bedrooms=0 rejected")

# Test 3: Invalid bedrooms (too high)
response = client.post("/validate", json={**valid_payload, "bedrooms": 20})
assert response.status_code == 422
print("✓ Test 3 passed: bedrooms=20 rejected")

# Test 4: Invalid sqft (too low)
response = client.post("/validate", json={**valid_payload, "sqft": 50})
assert response.status_code == 422
print("✓ Test 4 passed: sqft=50 rejected")

# Test 5: Invalid sqft (too high)
response = client.post("/validate", json={**valid_payload, "sqft": 50000})
assert response.status_code == 422
print("✓ Test 5 passed: sqft=50000 rejected")

# Test 6: Type coercion works
response = client.post("/validate", json={**valid_payload, "sqft": "1800"})  # string
assert response.status_code == 200  # Pydantic coerces "1800" → 1800.0
assert response.json()["validated"]["sqft"] == 1800.0
print("✓ Test 6 passed: Type coercion (string → float)")

print("\n✅ All tests passed!")

**Explanation:**

- `Field(ge=x, le=y)` enforces greater-or-equal and less-or-equal constraints
- Pydantic validates BEFORE your function runs — invalid data never reaches your code
- HTTP 422 is automatic; FastAPI includes detailed error messages showing which field failed
- `model_dump()` converts the Pydantic model back to a plain dict
- Pydantic coerces types safely: `"3"` → `3`, `"3.5"` → `3.5`, but `"abc"` fails

**Best practice:** Define constraints based on physical reality (bedrooms can't be 0 or 1000) and data types (year_built must be int). These constraints prevent garbage from reaching your model.

### Solution 2.3: Serving a Classification Model

In [ ]:
# Solution 2.3
# Training data
X_train = [[2, 0.6], [4, 0.8], [1, 0.5], [6, 0.9], [3, 0.7], [5, 0.85]]
y_train = [0, 1, 0, 1, 0, 1]

# Train model ONCE at module level
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

MODEL_VERSION = "1.0.0"


# Pydantic request model
class StudentFeatures(BaseModel):
    hours_studied: float = Field(ge=0, le=80)
    attendance: float = Field(ge=0, le=1)


# FastAPI app
app = FastAPI()


@app.post("/predict")
def predict(features: StudentFeatures):
    # Prepare input
    X = [[features.hours_studied, features.attendance]]
    
    # Get prediction
    proba = model.predict_proba(X)[0][1]  # probability of class 1 (pass)
    
    # Convert NumPy types to Python types
    return {
        "probability": round(float(proba), 4),
        "prediction": bool(proba >= 0.5),
        "model_version": MODEL_VERSION
    }


# Tests
client = TestClient(app)

# Test 1: Valid prediction
response = client.post("/predict", json={"hours_studied": 5.0, "attendance": 0.85})
assert response.status_code == 200
data = response.json()
print(f"Prediction for 5 hours, 85% attendance: {data}")

# Test 2: Response structure
assert set(data.keys()) == {"probability", "prediction", "model_version"}
print("✓ Test 2 passed: Response has correct keys")

# Test 3: Response types
assert isinstance(data["probability"], float)
assert isinstance(data["prediction"], bool)
assert isinstance(data["model_version"], str)
print("✓ Test 3 passed: Response types are correct")

# Test 4: Probability bounds
assert 0.0 <= data["probability"] <= 1.0
print("✓ Test 4 passed: Probability in [0, 1]")

# Test 5: Invalid hours (negative)
response = client.post("/predict", json={"hours_studied": -1, "attendance": 0.8})
assert response.status_code == 422
print("✓ Test 5 passed: Negative hours rejected")

# Test 6: Invalid attendance (out of range)
response = client.post("/predict", json={"hours_studied": 5, "attendance": 1.5})
assert response.status_code == 422
print("✓ Test 6 passed: Attendance > 1.0 rejected")

# Test 7: Boundary values
response = client.post("/predict", json={"hours_studied": 0, "attendance": 0})
assert response.status_code == 200
print("✓ Test 7 passed: Boundary values (0, 0) accepted")

response = client.post("/predict", json={"hours_studied": 80, "attendance": 1.0})
assert response.status_code == 200
print("✓ Test 8 passed: Boundary values (80, 1.0) accepted")

print("\n✅ All tests passed!")

**Explanation:**

- **Model trained once** at module level — no repeated training per request
- **Feature array construction**: `[[hours, attendance]]` — double brackets for 2D array
- **NumPy conversion**: `float()` and `bool()` convert NumPy types to JSON-serializable Python types
- **Model version**: Included in every response for auditability
- **Validation automatic**: Pydantic rejects negative hours and attendance > 1.0 with 422

**Best practice:** The endpoint is a thin wrapper: validate input → transform → call model → transform output. No business logic mixed with ML logic.

### Solution 2.4: Health Check Endpoint

In [ ]:
# Solution 2.4
MODEL_VERSION = "1.0.0"
FRAMEWORK = "sklearn"

# Simulate model loading
model = LogisticRegression()  # Just for demonstration
MODEL_LOADED = model is not None

app = FastAPI()


@app.get("/health")
def health():
    """Health check endpoint for load balancers and orchestrators."""
    return {
        "status": "ok",
        "model_version": MODEL_VERSION,
        "model_loaded": MODEL_LOADED
    }


@app.get("/version")
def version():
    """Return API and model version information."""
    return {
        "version": MODEL_VERSION,
        "framework": FRAMEWORK
    }


@app.post("/predict")
def predict():
    """Dummy prediction endpoint."""
    return {"prediction": "dummy"}


# Tests
client = TestClient(app)

# Test 1: Health check returns 200
response = client.get("/health")
assert response.status_code == 200
print("✓ Test 1 passed: Health check returns 200")

# Test 2: Health check has correct status
data = response.json()
assert data["status"] == "ok"
assert data["model_loaded"] is True
print("✓ Test 2 passed: Status is 'ok' and model_loaded is True")

# Test 3: Health check includes model version
assert "model_version" in data
assert data["model_version"] == "1.0.0"
print("✓ Test 3 passed: Model version included")

# Test 4: Version endpoint
response = client.get("/version")
assert response.status_code == 200
data = response.json()
assert "version" in data
assert "framework" in data
assert data["framework"] == "sklearn"
print("✓ Test 4 passed: Version endpoint works")

# Test 5: Predict endpoint exists
response = client.post("/predict")
assert response.status_code == 200
print("✓ Test 5 passed: Predict endpoint responds")

print("\n✅ All tests passed!")
print("\n💡 Real-world usage:")
print("   - Kubernetes liveness probe: GET /health every 10s")
print("   - Load balancer: only route traffic if status='ok'")
print("   - Incident triage: first question is 'which model version?'")

**Explanation:**

- **`/health`**: The endpoint orchestrators call to decide if this instance is ready
  - Return 200 + `status: "ok"` when ready to serve traffic
  - Return 503 or non-ok status if the model failed to load or dependencies are down
  - Keep it cheap — don't call the model, just check it loaded

- **`/version`**: Human-readable metadata for debugging
  - Which model version is deployed?
  - What framework/dependencies?

- **Why it matters**: Without `/health`, orchestrators can't tell if your service is alive or dead. They'll keep sending traffic to broken instances.

**Best practice:** Every production API needs `/health`. It's the first endpoint you add, not the last.

### Solution 2.5: Multi-Feature Regression API

In [ ]:
# Solution 2.5
# Training data
X_train = [[1500, 3, 2], [2000, 4, 3], [1200, 2, 1], [2500, 4, 3], [1800, 3, 2]]
y_train = [300000, 400000, 250000, 500000, 350000]

# Train model
regression_model = LinearRegression()
regression_model.fit(X_train, y_train)

MODEL_VERSION = "1.0.0"


# Pydantic model
class HouseFeatures(BaseModel):
    sqft: float = Field(ge=100, le=10000, description="Square footage")
    bedrooms: int = Field(ge=1, le=10, description="Number of bedrooms")
    bathrooms: float = Field(ge=0.5, le=8.0, description="Number of bathrooms")


app = FastAPI()


@app.post("/predict")
def predict(features: HouseFeatures):
    # Prepare input (order matters: sqft, bedrooms, bathrooms)
    X = [[features.sqft, features.bedrooms, features.bathrooms]]
    
    # Get prediction
    price = regression_model.predict(X)[0]  # returns np.float64
    
    # Convert NumPy types to Python types and format
    return {
        "predicted_price": round(float(price), 2),
        "input_features": features.model_dump(),
        "model_version": MODEL_VERSION
    }


# Tests
client = TestClient(app)

# Test 1: Valid input
valid_input = {"sqft": 1800, "bedrooms": 3, "bathrooms": 2.5}
response = client.post("/predict", json=valid_input)
assert response.status_code == 200
data = response.json()
print(f"Prediction: ${data['predicted_price']:,.2f}")
print("✓ Test 1 passed: Valid input accepted")

# Test 2: Response structure
assert "predicted_price" in data
assert "input_features" in data
assert "model_version" in data
print("✓ Test 2 passed: Response has all required fields")

# Test 3: Echo input features
assert data["input_features"] == valid_input
print("✓ Test 3 passed: Input features echoed correctly")

# Test 4: Price is positive
assert data["predicted_price"] > 0
print("✓ Test 4 passed: Predicted price is positive")

# Test 5: Invalid sqft (too low)
response = client.post("/predict", json={**valid_input, "sqft": 50})
assert response.status_code == 422
print("✓ Test 5 passed: sqft=50 rejected")

# Test 6: Invalid sqft (too high)
response = client.post("/predict", json={**valid_input, "sqft": 20000})
assert response.status_code == 422
print("✓ Test 6 passed: sqft=20000 rejected")

# Test 7: Invalid bedrooms
response = client.post("/predict", json={**valid_input, "bedrooms": 0})
assert response.status_code == 422
print("✓ Test 7 passed: bedrooms=0 rejected")

# Test 8: Invalid bathrooms
response = client.post("/predict", json={**valid_input, "bathrooms": 0.2})
assert response.status_code == 422
print("✓ Test 8 passed: bathrooms=0.2 rejected")

# Test 9: Boundary values
response = client.post("/predict", json={"sqft": 100, "bedrooms": 1, "bathrooms": 0.5})
assert response.status_code == 200
print("✓ Test 9 passed: Minimum boundary values accepted")

response = client.post("/predict", json={"sqft": 10000, "bedrooms": 10, "bathrooms": 8.0})
assert response.status_code == 200
print("✓ Test 10 passed: Maximum boundary values accepted")

print("\n✅ All tests passed!")

**Explanation:**

- **Feature order matters**: The model was trained on `[sqft, bedrooms, bathrooms]` — prediction must use the same order
- **Echo input features**: Including the input in the response creates an audit trail
- **Rounding**: `round(price, 2)` gives dollar-and-cents precision
- **NumPy conversion**: `.predict()` returns `np.ndarray`, indexing gives `np.float64`, which must be converted to `float`
- **Validation**: Constraints prevent nonsense like 0 bedrooms or 50 sqft

**Best practice:** Prediction responses should be self-describing. If someone finds `{"predicted_price": 350000}` in a log file, they should be able to understand what inputs produced it without digging through other logs.

### Solution 2.6: Error Handling with Custom Validation

In [ ]:
# Solution 2.6
class CreditCardFeatures(BaseModel):
    credit_limit: float = Field(ge=500, le=50000)
    current_balance: float = Field(ge=0, le=50000)
    payment_history_months: int = Field(ge=1, le=120)
    
    @validator('current_balance')
    def balance_must_not_exceed_limit(cls, v, values):
        # 'values' contains previously validated fields
        if 'credit_limit' in values and v > values['credit_limit']:
            raise ValueError(
                f"current_balance ({v}) cannot exceed credit_limit ({values['credit_limit']})"
            )
        return v


app = FastAPI()


@app.post("/validate")
def validate_credit(features: CreditCardFeatures):
    return {"validated": features.model_dump()}


# Tests
client = TestClient(app)

# Test 1: Valid data
valid_data = {
    "credit_limit": 10000,
    "current_balance": 5000,
    "payment_history_months": 24
}
response = client.post("/validate", json=valid_data)
assert response.status_code == 200
print("✓ Test 1 passed: Valid data accepted")

# Test 2: Balance equals limit (boundary)
response = client.post("/validate", json={
    "credit_limit": 10000,
    "current_balance": 10000,  # Equal is OK
    "payment_history_months": 24
})
assert response.status_code == 200
print("✓ Test 2 passed: Balance equals limit accepted")

# Test 3: Balance exceeds limit (custom validation failure)
response = client.post("/validate", json={
    "credit_limit": 10000,
    "current_balance": 15000,  # Exceeds limit!
    "payment_history_months": 24
})
assert response.status_code == 422
error_detail = response.json()["detail"]
print(f"✓ Test 3 passed: Balance > limit rejected")
print(f"  Error message: {error_detail[0]['msg']}")

# Test 4: Credit limit out of range
response = client.post("/validate", json={
    "credit_limit": 100,  # Below 500
    "current_balance": 50,
    "payment_history_months": 24
})
assert response.status_code == 422
print("✓ Test 4 passed: credit_limit < 500 rejected")

# Test 5: Negative balance
response = client.post("/validate", json={
    "credit_limit": 10000,
    "current_balance": -100,
    "payment_history_months": 24
})
assert response.status_code == 422
print("✓ Test 5 passed: Negative balance rejected")

# Test 6: Payment history out of range
response = client.post("/validate", json={
    "credit_limit": 10000,
    "current_balance": 5000,
    "payment_history_months": 0  # Below 1
})
assert response.status_code == 422
print("✓ Test 6 passed: payment_history_months=0 rejected")

print("\n✅ All tests passed!")
print("\n💡 Custom validators enforce business rules beyond simple type/range checks.")

**Explanation:**

- **`@validator` decorator**: Defines custom validation logic beyond simple type/range checks
- **Cross-field validation**: The validator checks `current_balance` against `credit_limit`
- **`values` parameter**: Contains previously validated fields (validators run in order)
- **`ValueError`**: Raising this exception triggers a 422 response with your custom message

**When to use custom validators:**
- Cross-field constraints (balance ≤ limit, end_date > start_date)
- Complex business rules (age must be 18+ for certain products)
- Format validation (phone numbers, email patterns beyond basic regex)

**Best practice:** Keep validators pure and focused. They should validate, not transform or fetch data. The error message should tell the client exactly what's wrong and how to fix it.

---
## Part 3: Challenge Problems

### Solution 3.1: Multi-Model API with Model Selection

In [ ]:
# Solution 3.1
# Training data
X_train = [[2, 0.6], [4, 0.8], [1, 0.5], [6, 0.9], [3, 0.7], [5, 0.85], [7, 0.95], [1.5, 0.55]]
y_train = [0, 1, 0, 1, 0, 1, 1, 0]

# Train both models at module level
logistic_model = LogisticRegression(max_iter=2000)
logistic_model.fit(X_train, y_train)

rf_model = RandomForestClassifier(n_estimators=10, random_state=42)
rf_model.fit(X_train, y_train)

MODEL_VERSION = "1.0.0"


# Pydantic model
class StudentFeatures(BaseModel):
    hours_studied: float = Field(ge=0, le=80)
    attendance: float = Field(ge=0, le=1)


# FastAPI app
app = FastAPI(title="Multi-Model Student Pass Predictor")


def make_prediction(model, model_type: str, features: StudentFeatures):
    """Shared prediction logic."""
    X = [[features.hours_studied, features.attendance]]
    proba = model.predict_proba(X)[0][1]
    
    return {
        "probability": round(float(proba), 4),
        "prediction": bool(proba >= 0.5),
        "model_type": model_type,
        "model_version": MODEL_VERSION
    }


@app.post("/predict/logistic")
def predict_logistic(features: StudentFeatures):
    return make_prediction(logistic_model, "logistic_regression", features)


@app.post("/predict/random-forest")
def predict_random_forest(features: StudentFeatures):
    return make_prediction(rf_model, "random_forest", features)


@app.get("/models")
def list_models():
    return {
        "models": [
            {
                "name": "logistic_regression",
                "endpoint": "/predict/logistic",
                "version": MODEL_VERSION
            },
            {
                "name": "random_forest",
                "endpoint": "/predict/random-forest",
                "version": MODEL_VERSION
            }
        ]
    }


# Tests
client = TestClient(app)

test_input = {"hours_studied": 5.0, "attendance": 0.85}

# Test 1: Logistic regression endpoint
response = client.post("/predict/logistic", json=test_input)
assert response.status_code == 200
data = response.json()
assert data["model_type"] == "logistic_regression"
assert "probability" in data
print(f"Logistic prediction: {data}")
print("✓ Test 1 passed: Logistic endpoint works")

# Test 2: Random forest endpoint
response = client.post("/predict/random-forest", json=test_input)
assert response.status_code == 200
data = response.json()
assert data["model_type"] == "random_forest"
assert "probability" in data
print(f"Random forest prediction: {data}")
print("✓ Test 2 passed: Random forest endpoint works")

# Test 3: Models may give different predictions
lr_response = client.post("/predict/logistic", json=test_input).json()
rf_response = client.post("/predict/random-forest", json=test_input).json()
print(f"Probability difference: {abs(lr_response['probability'] - rf_response['probability']):.4f}")
print("✓ Test 3 passed: Both models return predictions")

# Test 4: List models endpoint
response = client.get("/models")
assert response.status_code == 200
data = response.json()
assert len(data["models"]) == 2
model_names = [m["name"] for m in data["models"]]
assert "logistic_regression" in model_names
assert "random_forest" in model_names
print("✓ Test 4 passed: List models endpoint works")

# Test 5: Both endpoints validate input
invalid_input = {"hours_studied": -5, "attendance": 0.8}
assert client.post("/predict/logistic", json=invalid_input).status_code == 422
assert client.post("/predict/random-forest", json=invalid_input).status_code == 422
print("✓ Test 5 passed: Both endpoints validate input")

# Test 6: Response structure is consistent
for endpoint in ["/predict/logistic", "/predict/random-forest"]:
    response = client.post(endpoint, json=test_input).json()
    assert set(response.keys()) == {"probability", "prediction", "model_type", "model_version"}
print("✓ Test 6 passed: Response structure is consistent")

print("\n✅ All tests passed!")
print("\n💡 Real-world: A/B testing framework would route traffic between endpoints")
print("   and compare conversion rates to choose the winner.")

**Explanation:**

- **Both models loaded once** at module level — no repeated loading
- **Shared prediction logic**: `make_prediction()` function eliminates duplication
- **Model type in response**: Clients know which model produced the prediction
- **Model registry endpoint**: `/models` lists what's available (self-documenting API)

**Real-world pattern:**
- Load multiple model versions or types
- A/B test between them by routing traffic
- Compare metrics (latency, accuracy, business KPIs)
- Gradually shift traffic to the winner

**Best practice:** When serving multiple models, make model selection explicit in the URL (`/predict/v1`, `/predict/logistic`) rather than in request body. URL-based routing is easier to cache, monitor, and route.

### Solution 3.2: Batch Prediction Endpoint

In [ ]:
# Solution 3.2
# Training
X_train = [[2, 0.6], [4, 0.8], [1, 0.5], [6, 0.9], [3, 0.7], [5, 0.85]]
y_train = [0, 1, 0, 1, 0, 1]

batch_model = LogisticRegression(max_iter=2000)
batch_model.fit(X_train, y_train)

MODEL_VERSION = "1.0.0"


# Pydantic models
class StudentFeatures(BaseModel):
    hours_studied: float = Field(ge=0, le=80)
    attendance: float = Field(ge=0, le=1)


class BatchRequest(BaseModel):
    samples: List[StudentFeatures] = Field(min_items=1, max_items=100)


# FastAPI app
app = FastAPI()


@app.post("/predict/batch")
def predict_batch(request: BatchRequest):
    # Convert all samples to feature matrix
    X = [[s.hours_studied, s.attendance] for s in request.samples]
    
    # Get predictions for all samples at once (vectorized)
    probas = batch_model.predict_proba(X)[:, 1]  # probability of class 1
    
    # Build response
    predictions = []
    for i, (sample, proba) in enumerate(zip(request.samples, probas)):
        predictions.append({
            "sample_id": i,
            "hours_studied": sample.hours_studied,
            "attendance": sample.attendance,
            "probability": round(float(proba), 4),
            "prediction": bool(proba >= 0.5)
        })
    
    return {
        "predictions": predictions,
        "count": len(predictions),
        "model_version": MODEL_VERSION
    }


# Tests
client = TestClient(app)

# Test 1: Single sample
response = client.post("/predict/batch", json={
    "samples": [{"hours_studied": 5.0, "attendance": 0.85}]
})
assert response.status_code == 200
data = response.json()
assert data["count"] == 1
assert len(data["predictions"]) == 1
print("✓ Test 1 passed: Single sample")

# Test 2: Multiple valid samples
response = client.post("/predict/batch", json={
    "samples": [
        {"hours_studied": 2.0, "attendance": 0.6},
        {"hours_studied": 5.0, "attendance": 0.85},
        {"hours_studied": 7.0, "attendance": 0.95}
    ]
})
assert response.status_code == 200
data = response.json()
assert data["count"] == 3
assert len(data["predictions"]) == 3
print(f"Batch predictions: {data['count']} samples processed")
print("✓ Test 2 passed: Multiple samples")

# Test 3: Each prediction has correct structure
for pred in data["predictions"]:
    assert "sample_id" in pred
    assert "probability" in pred
    assert "prediction" in pred
    assert isinstance(pred["probability"], float)
    assert isinstance(pred["prediction"], bool)
print("✓ Test 3 passed: Prediction structure correct")

# Test 4: Empty list rejected
response = client.post("/predict/batch", json={"samples": []})
assert response.status_code == 422
print("✓ Test 4 passed: Empty list rejected")

# Test 5: Over-limit list rejected
too_many_samples = [{"hours_studied": 5.0, "attendance": 0.8} for _ in range(101)]
response = client.post("/predict/batch", json={"samples": too_many_samples})
assert response.status_code == 422
print("✓ Test 5 passed: >100 samples rejected")

# Test 6: Invalid sample in batch is rejected
response = client.post("/predict/batch", json={
    "samples": [
        {"hours_studied": 5.0, "attendance": 0.85},
        {"hours_studied": -1, "attendance": 0.8}  # Invalid!
    ]
})
assert response.status_code == 422
print("✓ Test 6 passed: Invalid sample in batch rejected")

# Test 7: Boundary values (1 and 100 samples)
response = client.post("/predict/batch", json={
    "samples": [{"hours_studied": 5.0, "attendance": 0.8}]
})
assert response.status_code == 200
print("✓ Test 7 passed: 1 sample accepted")

exactly_100 = [{"hours_studied": 5.0, "attendance": 0.8} for _ in range(100)]
response = client.post("/predict/batch", json={"samples": exactly_100})
assert response.status_code == 200
assert response.json()["count"] == 100
print("✓ Test 8 passed: 100 samples accepted")

print("\n✅ All tests passed!")
print("\n💡 Batch endpoints are useful for:")
print("   - Processing uploaded files (CSV with 50 rows)")
print("   - Reducing HTTP overhead (1 request vs 100)")
print("   - Vectorized inference (model processes all at once)")

**Explanation:**

- **`List[StudentFeatures]`**: Each item in the list is validated individually
- **`min_items` and `max_items`**: Pydantic enforces list length constraints
- **Vectorized inference**: Convert all samples to a matrix, call `.predict_proba()` once — much faster than looping
- **Sample ID**: Each prediction includes its index for client-side matching
- **Echo inputs**: Predictions include the input features for auditability

**Why 100 sample limit?**
- Prevents abuse (someone sending 1 million samples)
- Keeps request/response sizes reasonable
- Keeps latency predictable
- For >100 samples, clients should use true batch scoring (file upload + async processing)

**Best practice:** Batch endpoints are a middle ground between single-prediction APIs and full batch jobs. They're perfect for moderate-size batches (10-100 items) where you want synchronous results.

### Solution 3.3: Full Production-Ready API

In [ ]:
# Solution 3.3 — Production-ready model serving API
# Training
X_train = np.array([[2, 0.6], [4, 0.8], [1, 0.5], [6, 0.9], [3, 0.7], [5, 0.85], [7, 0.95]])
y_train = np.array([0, 1, 0, 1, 0, 1, 1])

# Build a proper pipeline with preprocessing
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=2000))
])
pipeline.fit(X_train, y_train)

# Metadata
MODEL_VERSION = "1.0.0"
API_VERSION = "1.0.0"
TRAINING_DATE = "2026-08-26"
FEATURE_NAMES = ["hours_studied", "attendance"]


# Pydantic models
class StudentFeatures(BaseModel):
    hours_studied: float = Field(
        ge=0, 
        le=80, 
        description="Weekly study hours"
    )
    attendance: float = Field(
        ge=0, 
        le=1, 
        description="Attendance rate (0-1)"
    )
    
    @validator('attendance')
    def round_attendance(cls, v):
        # Ensure precision doesn't exceed what's meaningful
        return round(v, 3)


# FastAPI app with metadata
app = FastAPI(
    title="Student Pass Prediction API",
    description="Production ML API for predicting student pass/fail outcomes",
    version=API_VERSION
)


@app.get("/")
def root():
    """Root endpoint with API information."""
    return {
        "service": "Student Pass Prediction API",
        "version": API_VERSION,
        "status": "operational",
        "endpoints": {
            "health": "/health",
            "docs": "/docs",
            "predict": "/predict",
            "version": "/version",
            "metadata": "/models/metadata"
        }
    }


@app.get("/health")
def health():
    """Health check for load balancers and orchestrators."""
    # In production, you might check:
    # - Model loaded successfully
    # - Database connection alive
    # - Disk space available
    model_loaded = pipeline is not None
    
    return {
        "status": "ok" if model_loaded else "error",
        "model_version": MODEL_VERSION,
        "model_loaded": model_loaded,
        "api_version": API_VERSION
    }


@app.get("/version")
def version():
    """Model and API version information."""
    return {
        "api_version": API_VERSION,
        "model_version": MODEL_VERSION,
        "framework": "scikit-learn",
        "model_type": "LogisticRegression with StandardScaler"
    }


@app.get("/models/metadata")
def model_metadata():
    """Detailed model training metadata."""
    return {
        "model_version": MODEL_VERSION,
        "training_date": TRAINING_DATE,
        "feature_names": FEATURE_NAMES,
        "n_features": len(FEATURE_NAMES),
        "model_type": "LogisticRegression",
        "preprocessing": ["StandardScaler"],
        "training_samples": len(X_train)
    }


@app.get("/models/explain")
def model_explain():
    """Model coefficients and feature importance."""
    # Extract the logistic regression from the pipeline
    lr = pipeline.named_steps['classifier']
    coefficients = lr.coef_[0]
    
    # Build feature importance dict
    feature_importance = [
        {
            "feature": name,
            "coefficient": round(float(coef), 4),
            "direction": "positive" if coef > 0 else "negative"
        }
        for name, coef in zip(FEATURE_NAMES, coefficients)
    ]
    
    # Sort by absolute coefficient value
    feature_importance.sort(key=lambda x: abs(x['coefficient']), reverse=True)
    
    return {
        "model_version": MODEL_VERSION,
        "feature_importance": feature_importance,
        "intercept": round(float(lr.intercept_[0]), 4)
    }


@app.post("/predict")
def predict(features: StudentFeatures):
    """Main prediction endpoint."""
    # Prepare input
    X = [[features.hours_studied, features.attendance]]
    
    # Get prediction
    proba = pipeline.predict_proba(X)[0][1]
    prediction = pipeline.predict(X)[0]
    
    # Convert NumPy types
    return {
        "prediction": bool(prediction),
        "probability": round(float(proba), 4),
        "confidence": round(float(max(proba, 1 - proba)), 4),  # max probability
        "input_features": {
            "hours_studied": features.hours_studied,
            "attendance": features.attendance
        },
        "model_version": MODEL_VERSION,
        "api_version": API_VERSION
    }


# Comprehensive test suite
client = TestClient(app)

print("Running comprehensive test suite...\n")

# Test 1: Root endpoint
response = client.get("/")
assert response.status_code == 200
data = response.json()
assert "endpoints" in data
print("✓ Test 1 passed: Root endpoint")

# Test 2: Health check
response = client.get("/health")
assert response.status_code == 200
data = response.json()
assert data["status"] == "ok"
assert data["model_loaded"] is True
print("✓ Test 2 passed: Health check")

# Test 3: Version endpoint
response = client.get("/version")
assert response.status_code == 200
data = response.json()
assert "model_version" in data
assert "api_version" in data
print("✓ Test 3 passed: Version endpoint")

# Test 4: Model metadata
response = client.get("/models/metadata")
assert response.status_code == 200
data = response.json()
assert data["feature_names"] == FEATURE_NAMES
assert data["n_features"] == 2
print("✓ Test 4 passed: Model metadata")

# Test 5: Model explanation
response = client.get("/models/explain")
assert response.status_code == 200
data = response.json()
assert "feature_importance" in data
assert len(data["feature_importance"]) == 2
print(f"Feature importance: {data['feature_importance']}")
print("✓ Test 5 passed: Model explanation")

# Test 6: Prediction happy path
response = client.post("/predict", json={"hours_studied": 5.0, "attendance": 0.85})
assert response.status_code == 200
data = response.json()
assert "prediction" in data
assert "probability" in data
assert "confidence" in data
assert "input_features" in data
print(f"Sample prediction: {data}")
print("✓ Test 6 passed: Prediction happy path")

# Test 7: Response types
assert isinstance(data["prediction"], bool)
assert isinstance(data["probability"], float)
assert isinstance(data["confidence"], float)
print("✓ Test 7 passed: Response types correct")

# Test 8: Probability and confidence bounds
assert 0.0 <= data["probability"] <= 1.0
assert 0.5 <= data["confidence"] <= 1.0  # confidence is max(p, 1-p)
print("✓ Test 8 passed: Probability and confidence in valid ranges")

# Test 9: Input features echoed
assert data["input_features"]["hours_studied"] == 5.0
assert data["input_features"]["attendance"] == 0.85
print("✓ Test 9 passed: Input features echoed")

# Test 10: Model version in response
assert data["model_version"] == MODEL_VERSION
print("✓ Test 10 passed: Model version in response")

# Test 11: Invalid hours rejected
response = client.post("/predict", json={"hours_studied": -1, "attendance": 0.8})
assert response.status_code == 422
print("✓ Test 11 passed: Negative hours rejected")

# Test 12: Invalid attendance rejected
response = client.post("/predict", json={"hours_studied": 5, "attendance": 1.5})
assert response.status_code == 422
print("✓ Test 12 passed: Attendance > 1.0 rejected")

# Test 13: Missing field rejected
response = client.post("/predict", json={"hours_studied": 5})
assert response.status_code == 422
print("✓ Test 13 passed: Missing field rejected")

# Test 14: Boundary values
response = client.post("/predict", json={"hours_studied": 0, "attendance": 0})
assert response.status_code == 200
print("✓ Test 14 passed: Minimum boundary values")

response = client.post("/predict", json={"hours_studied": 80, "attendance": 1.0})
assert response.status_code == 200
print("✓ Test 15 passed: Maximum boundary values")

# Test 16: Type coercion
response = client.post("/predict", json={"hours_studied": "5.0", "attendance": "0.85"})
assert response.status_code == 200
print("✓ Test 16 passed: String to float coercion")

# Test 17: Multiple predictions are consistent
test_input = {"hours_studied": 5.0, "attendance": 0.85}
response1 = client.post("/predict", json=test_input).json()
response2 = client.post("/predict", json=test_input).json()
assert response1["probability"] == response2["probability"]
print("✓ Test 17 passed: Predictions are deterministic")

# Test 18: Edge case - very low hours
response = client.post("/predict", json={"hours_studied": 0.1, "attendance": 0.3})
assert response.status_code == 200
data = response.json()
print(f"Low hours prediction: {data['prediction']} (prob={data['probability']})")
print("✓ Test 18 passed: Low hours prediction")

# Test 19: Edge case - very high hours
response = client.post("/predict", json={"hours_studied": 75, "attendance": 1.0})
assert response.status_code == 200
data = response.json()
print(f"High hours prediction: {data['prediction']} (prob={data['probability']})")
print("✓ Test 19 passed: High hours prediction")

print("\n" + "="*70)
print("✅ ALL 19 TESTS PASSED!")
print("="*70)

print("\n📋 API Summary:")
print(f"   API Version: {API_VERSION}")
print(f"   Model Version: {MODEL_VERSION}")
print(f"   Endpoints: 7 (/, /health, /version, /models/*, /predict)")
print(f"   Features: {FEATURE_NAMES}")
print(f"   Training Date: {TRAINING_DATE}")

print("\n🏗️ Production Readiness Checklist:")
print("   ✓ Health check endpoint")
print("   ✓ Version information")
print("   ✓ Model metadata exposed")
print("   ✓ Feature importance/explainability")
print("   ✓ Comprehensive input validation")
print("   ✓ Proper error handling (422 for validation)")
print("   ✓ NumPy type conversion")
print("   ✓ Input echo for auditability")
print("   ✓ Model version in every response")
print("   ✓ Contract tests cover happy/sad/edge paths")
print("   ✓ Pipeline with preprocessing (StandardScaler)")
print("   ✓ Confidence scores included")
print("   ✓ Self-documenting root endpoint")

**Explanation — Production Best Practices:**

1. **Pipeline with preprocessing**: StandardScaler ensures consistent feature transformation
2. **Rich metadata endpoints**: `/models/metadata` and `/models/explain` support debugging and auditing
3. **Feature importance**: Helps stakeholders understand what drives predictions
4. **Confidence score**: `max(p, 1-p)` tells you how certain the model is
5. **Input echo**: Every response includes the inputs that produced it (audit trail)
6. **Version everywhere**: Model and API versions in responses enable incident triage
7. **Comprehensive validation**: Pydantic + custom validators enforce all constraints
8. **Self-documenting**: Root endpoint lists all available endpoints
9. **Health checks**: Load balancers know when the service is ready
10. **Extensive tests**: 19 tests covering happy paths, boundaries, errors, and edge cases

**Real-world deployment:**
```dockerfile
FROM python:3.12-slim
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY model_pipeline.joblib app.py .
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

Then deploy to Kubernetes with:
- Liveness probe: `GET /health`
- Readiness probe: `GET /health`
- Prometheus metrics scraping `/metrics` (add with `prometheus-fastapi-instrumentator`)
- Horizontal autoscaling based on CPU/request rate

**This is a production-ready model serving API.**

---
## Key Takeaways

### 1. Model Serving Fundamentals
- Load models **once** at module level, not per-request
- Validate input with Pydantic models — bad data dies at the border with 422
- Convert NumPy types to Python types before returning JSON
- Echo model version in every response for auditability

### 2. API Design
- `/health` endpoint is mandatory for orchestrators
- Include version and metadata endpoints for debugging
- Echo input features in prediction responses
- Return confidence scores, not just predictions
- Use URL paths for model selection (`/predict/v1`, `/predict/logistic`)

### 3. Validation
- Define constraints based on physical reality (hours ≥ 0, attendance ≤ 1)
- Use `Field(ge=, le=)` for range constraints
- Use custom `@validator` for cross-field rules
- Trust Pydantic to reject bad input — your endpoint only sees valid data

### 4. Testing
- TestClient gives you full API testing with zero servers
- Test happy path, boundaries, invalid input, missing fields, and edge cases
- Contract tests belong in CI — they guard the API promise
- Verify response structure, types, and value ranges

### 5. Production Readiness
- Health checks for liveness/readiness probes
- Metadata endpoints for incident triage
- Feature importance/explainability for stakeholder trust
- Comprehensive error handling (never return 500 for bad input)
- Pipeline with preprocessing for consistent feature transformation
- Extensive test coverage (19+ tests minimum)

### 6. Real-World Patterns
- Batch endpoints for moderate-size batches (1-100 samples)
- Multi-model serving for A/B testing
- Version URLs for safe model evolution
- Log every request/response for monitoring (Lesson 06)
- Container deployment with health checks

**Bottom line:** A production ML API is a validated, versioned, tested, monitored contract between your model and the world. FastAPI + Pydantic + TestClient make that contract enforceable and testable.